In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    RocCurveDisplay,
    PrecisionRecallDisplay
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


In [ ]:

test_df = pd.read_csv(
    "../data/processed/test_data.csv",
    parse_dates=["snapshot_date"]
)

selected_model = joblib.load(
    "../models/logistic_regression_pipeline.pkl"
)

with open("../models/model_metadata.json", "r") as file:
    model_metadata = json.load(file)

selected_threshold = model_metadata["classification_threshold"]

print("Test dataset shape:", test_df.shape)
print("Selected model:", model_metadata["selected_model"])
print("Classification threshold:", selected_threshold)

print("\nTest period:")
print(test_df["snapshot_date"].min(), "to", test_df["snapshot_date"].max())

In [ ]:
feature_columns = (
    model_metadata["numerical_features"]
    + model_metadata["categorical_features"]
)

missing_columns = set(feature_columns + ["churn"]) - set(test_df.columns)

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

X_test = test_df[feature_columns].copy()
y_test = test_df["churn"].copy()

print("Number of features:", len(feature_columns))
print("Test feature shape:", X_test.shape)
print("Test target shape:", y_test.shape)

In [ ]:

y_test_probability = selected_model.predict_proba(X_test)[:, 1]

y_test_pred = (
    y_test_probability >= selected_threshold
).astype(int)

print("Predictions generated successfully.")
print("Total observations:", len(y_test))
print("Predicted churners:", y_test_pred.sum())
print("Predicted non-churners:", (y_test_pred == 0).sum())

In [ ]:
test_metrics = {
    "Accuracy": accuracy_score(y_test, y_test_pred),
    "Precision": precision_score(y_test, y_test_pred, zero_division=0),
    "Recall": recall_score(y_test, y_test_pred, zero_division=0),
    "F1 Score": f1_score(y_test, y_test_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_test_probability),
    "PR-AUC": average_precision_score(y_test, y_test_probability)
}

test_results_df = pd.DataFrame(
    [test_metrics],
    index=["Logistic Regression"]
)

print("Final Test Results:")
print(test_results_df.round(4))

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)

plt.figure(figsize=(7, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Non-Churn", "Churn"],
    yticklabels=["Non-Churn", "Churn"]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Logistic Regression")
plt.tight_layout()
plt.show()

print("Confusion Matrix:")
print(cm)

In [ ]:
print(
    classification_report(
        y_test,
        y_test_pred,
        target_names=["Non-Churn", "Churn"],
        zero_division=0
    )
)

In [ ]:
RocCurveDisplay.from_predictions(
    y_test,
    y_test_probability
)

plt.title("ROC Curve - Logistic Regression")
plt.tight_layout()
plt.show()

In [ ]:
PrecisionRecallDisplay.from_predictions(
    y_test,
    y_test_probability
)

plt.title("Precision-Recall Curve - Logistic Regression")
plt.tight_layout()
plt.show()

In [ ]:
validation_df = pd.read_csv(
    "../data/processed/validation_data.csv",
    parse_dates=["snapshot_date"]
)

missing_columns = set(feature_columns + ["churn"]) - set(validation_df.columns)

if missing_columns:
    raise ValueError(f"Missing validation columns: {missing_columns}")

X_validation = validation_df[feature_columns].copy()
y_validation = validation_df["churn"].copy()

y_validation_probability = selected_model.predict_proba(
    X_validation
)[:, 1]

y_validation_pred = (
    y_validation_probability >= selected_threshold
).astype(int)

validation_metrics = {
    "Accuracy": accuracy_score(y_validation, y_validation_pred),
    "Precision": precision_score(y_validation, y_validation_pred, zero_division=0),
    "Recall": recall_score(y_validation, y_validation_pred, zero_division=0),
    "F1 Score": f1_score(y_validation, y_validation_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_validation, y_validation_probability),
    "PR-AUC": average_precision_score(y_validation, y_validation_probability)
}

print("Validation Results:")
print(pd.Series(validation_metrics).round(4))

In [ ]:
comparison_df = pd.DataFrame({
    "Metric": list(validation_metrics.keys()),
    "Validation": list(validation_metrics.values()),
    "Test": [test_metrics[metric] for metric in validation_metrics]
})

comparison_df["Difference"] = (
    comparison_df["Test"] - comparison_df["Validation"]
)

print("Validation vs Test Comparison:")
print(comparison_df.round(4).to_string(index=False))

In [ ]:
os.makedirs("../reports/figures", exist_ok=True)

test_results_df.to_csv(
    "../reports/final_test_results.csv"
)

comparison_df.to_csv(
    "../reports/validation_test_comparison.csv",
    index=False
)

print("Evaluation tables saved successfully.")

In [ ]:
x = np.arange(len(comparison_df))
width = 0.35

plt.figure(figsize=(11, 6))

plt.bar(
    x - width / 2,
    comparison_df["Validation"],
    width,
    label="Validation"
)

plt.bar(
    x + width / 2,
    comparison_df["Test"],
    width,
    label="Test"
)

plt.xticks(x, comparison_df["Metric"], rotation=20)
plt.ylabel("Score")
plt.title("Validation vs Test Performance")
plt.legend()
plt.tight_layout()


plt.show()